# Portable Laptop QC Notebook

Cleaned workflow for local QC, Z-selection testing, 2D object QC scoring, lightweight napari inspection, and final area/N/C plotting from the exported bundle.

## 1. User settings

In [ ]:

from pathlib import Path

# --- local paths ---
BUNDLE_DIR = Path(r"C:/Users/cowboy/OneDrive/Documents/University of Alabama/Nuclear_Scaling/Data_Sets/Control/portable_laptop_bundle")
RAW_IMAGE_PATH = None  # optional local raw image path

# --- acquisition assumptions ---
PIXEL_SIZE_UM = 0.162
N_FOV = 6
SERPENTINE = True

# --- default napari view ---
NAPARI_T_IDX = 0
NAPARI_SHOW_ALL_Z = True
AUTO_OPEN_NAPARI = False

# --- object QC defaults ---
QC_MIN_AREA_UM2 = 120
QC_MAX_AREA_UM2 = 600
QC_MIN_FILL_FRACTION = 0.60
QC_MIN_CIRCULARITY = 0.55
QC_MIN_MEAN_PROB = 0.40
QC_MIN_MAX_PROB = 0.60
QC_SCORE_THRESHOLD = 0.35

# --- grouped object filtering defaults ---
MIN_AREA_PX_GROUP = 4000
MIN_FILL_FRACTION_GROUP = 0.70
EDGE_BUFFER_PX = 5

# --- 3D grouping defaults ---
GROUP_MAX_DIST_PX = 20
MIN_Z_SLICES_VALID = 3


## 2. Imports and helper functions

In [ ]:

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tiff


def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)


def load_table_prefer_parquet_csv_pickle(stem: Path):
    parquet_path = stem.with_suffix(".parquet")
    csv_path = stem.with_suffix(".csv")
    pkl_path = stem.with_suffix(".pkl")

    if parquet_path.exists():
        return pd.read_parquet(parquet_path), parquet_path
    if csv_path.exists():
        return pd.read_csv(csv_path), csv_path
    if pkl_path.exists():
        return pd.read_pickle(pkl_path), pkl_path
    return None, None


def safe_read_tiff(path: Path, memmap_preferred: bool = True):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None
    if memmap_preferred:
        try:
            return tiff.memmap(path)
        except Exception:
            pass
    return tiff.imread(path)


def describe_df(df: pd.DataFrame, name: str, n: int = 5):
    if df is None:
        print(f"{name}: missing")
        return
    print(f"{name}: shape={df.shape}")
    print(f"Columns: {list(df.columns)}")
    display(df.head(n))


def ensure_area_um2(df: pd.DataFrame, pixel_size_um: float, area_px_col: str = "area_px", area_um2_col: str = "area_um2"):
    if df is None or df.empty:
        return df
    out = df.copy()
    if area_um2_col not in out.columns:
        if area_px_col not in out.columns:
            raise ValueError(f"Missing both '{area_um2_col}' and '{area_px_col}'")
        out[area_um2_col] = out[area_px_col] * (pixel_size_um ** 2)
    return out


def reconstruct_true_time_from_x(df: pd.DataFrame, n_fov: int = 6, serpentine: bool = True):
    if df is None or df.empty:
        return df
    required = ["t", "centroid_x_px"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for time reconstruction: {missing}")

    out = df.copy()
    max_x = out["centroid_x_px"].max()
    if pd.isna(max_x) or max_x <= 0:
        raise ValueError("Could not infer FOV width from centroid_x_px")

    fov_width = max_x / n_fov
    out["fov_id"] = (out["centroid_x_px"] // fov_width).astype(int)
    out["fov_id"] = out["fov_id"].clip(0, n_fov - 1)

    if serpentine:
        reverse_mask = out["t"] % 2 == 1
        out.loc[reverse_mask, "fov_id"] = (n_fov - 1) - out.loc[reverse_mask, "fov_id"]

    out["true_time_min"] = out["t"] * n_fov + out["fov_id"]
    return out


def add_fill_fraction(df: pd.DataFrame):
    if df is None or df.empty:
        return df
    needed = ["bbox_min_row", "bbox_min_col", "bbox_max_row", "bbox_max_col", "area_px"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns for fill_fraction: {missing}")

    out = df.copy()
    bbox_height = (out["bbox_max_row"] - out["bbox_min_row"]).clip(lower=1)
    bbox_width = (out["bbox_max_col"] - out["bbox_min_col"]).clip(lower=1)
    out["bbox_area_px"] = bbox_height * bbox_width
    out["fill_fraction"] = out["area_px"] / out["bbox_area_px"]
    return out


def add_edge_touching_from_bbox(df: pd.DataFrame, edge_buffer_px: int = 5):
    if df is None or df.empty:
        return df
    out = df.copy()
    bbox_cols = ["bbox_min_row", "bbox_min_col", "bbox_max_row", "bbox_max_col"]
    missing = [c for c in bbox_cols if c not in out.columns]
    if missing:
        out["is_edge_touching"] = False
        return out

    image_height_px = out["bbox_max_row"].max()
    image_width_px = out["bbox_max_col"].max()

    out["is_edge_touching"] = (
        (out["bbox_min_row"] <= edge_buffer_px) |
        (out["bbox_min_col"] <= edge_buffer_px) |
        (out["bbox_max_row"] >= image_height_px - edge_buffer_px) |
        (out["bbox_max_col"] >= image_width_px - edge_buffer_px)
    )
    return out


def clean_grouped_objects_before_best_z(
    df: pd.DataFrame,
    min_area_px: float = 4000,
    min_fill_fraction: float = 0.70,
    min_circularity: float | None = None,
    max_area_um2: float | None = 600,
    pixel_size_um: float = 0.162,
    remove_edge_touching: bool = True,
    edge_buffer_px: int = 5,
):
    if df is None or df.empty:
        print("No data to clean.")
        return df.copy()

    out = df.copy()
    if "area_px" not in out.columns:
        raise ValueError("Missing required column 'area_px'")

    out = out[out["area_px"] >= min_area_px].copy()
    out = add_fill_fraction(out)

    if min_fill_fraction is not None:
        out = out[out["fill_fraction"] >= min_fill_fraction].copy()

    if min_circularity is not None and "circularity" in out.columns:
        out = out[out["circularity"] >= min_circularity].copy()

    if remove_edge_touching:
        if "is_edge_touching" not in out.columns:
            out = add_edge_touching_from_bbox(out, edge_buffer_px=edge_buffer_px)
        out = out[~out["is_edge_touching"]].copy()

    out = ensure_area_um2(out, pixel_size_um)
    if max_area_um2 is not None:
        out = out[out["area_um2"] <= max_area_um2].copy()

    print("Rows after grouped-object cleaning:", len(out))
    return out


def group_objects_across_z(df: pd.DataFrame, max_dist_px: float = 20):
    df = df.copy()
    df = df.sort_values(["t", "z"]).reset_index(drop=True)
    df["nucleus_3d_id"] = -1
    current_id = 0

    for _, t_df in df.groupby("t"):
        t_df = t_df.sort_values("z")
        prev_slice = None

        for _, z_df in t_df.groupby("z"):
            z_df = z_df.copy()

            if prev_slice is None:
                for idx in z_df.index:
                    df.loc[idx, "nucleus_3d_id"] = current_id
                    current_id += 1
                prev_slice = z_df
                continue

            for idx, row in z_df.iterrows():
                x, y = row["centroid_x_px"], row["centroid_y_px"]

                dists = np.sqrt(
                    (prev_slice["centroid_x_px"] - x) ** 2 +
                    (prev_slice["centroid_y_px"] - y) ** 2
                )

                if len(dists) > 0 and dists.min() < max_dist_px:
                    match_idx = prev_slice.index[dists.argmin()]
                    df.loc[idx, "nucleus_3d_id"] = df.loc[match_idx, "nucleus_3d_id"]
                else:
                    df.loc[idx, "nucleus_3d_id"] = current_id
                    current_id += 1

            prev_slice = z_df

    return df


def keep_valid_nuclei(df: pd.DataFrame, min_z_slices: int = 3):
    if df is None or df.empty:
        print("No data to validate.")
        return df.copy()

    if "nucleus_3d_id" not in df.columns:
        raise ValueError("Missing required column 'nucleus_3d_id'")

    valid_ids = (
        df.groupby("nucleus_3d_id")["z"]
        .nunique()
        .loc[lambda s: s >= min_z_slices]
        .index
    )
    out = df[df["nucleus_3d_id"].isin(valid_ids)].copy()

    print("Total grouped rows:", len(df))
    print("Valid grouped rows:", len(out))
    print("Valid nuclei count:", out["nucleus_3d_id"].nunique())
    return out


def select_best_z_by_composite_score(
    df: pd.DataFrame,
    group_col: str = "nucleus_3d_id",
    z_col: str = "z",
    area_col: str = "area_px",
    circularity_col: str = "circularity",
    fill_fraction_col: str = "fill_fraction",
    edge_col: str = "is_edge_touching",
    min_area_px: float = 20,
    area_weight: float = 0.45,
    circularity_weight: float = 0.25,
    fill_fraction_weight: float = 0.30,
    edge_penalty: float = 1.0,
):
    if df is None or df.empty:
        return df.copy()

    required = [group_col, z_col, area_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    work = df.copy()
    if circularity_col not in work.columns:
        work[circularity_col] = 0.0
    if fill_fraction_col not in work.columns:
        work[fill_fraction_col] = 0.0
    if edge_col not in work.columns:
        work[edge_col] = False

    work = work[work[area_col] >= min_area_px].copy()
    if work.empty:
        return work

    def _normalize(series: pd.Series) -> pd.Series:
        smin = series.min()
        smax = series.max()
        if pd.isna(smin) or pd.isna(smax) or smax == smin:
            return pd.Series(np.zeros(len(series)), index=series.index)
        return (series - smin) / (smax - smin)

    selected_rows = []
    for _, g in work.groupby(group_col, dropna=False):
        g = g.copy()
        g["area_norm"] = _normalize(g[area_col].astype(float))
        g["circ_norm"] = _normalize(g[circularity_col].astype(float))
        g["fill_norm"] = _normalize(g[fill_fraction_col].astype(float))
        g["edge_penalty_val"] = g[edge_col].astype(float) * edge_penalty
        g["z_score"] = (
            area_weight * g["area_norm"] +
            circularity_weight * g["circ_norm"] +
            fill_fraction_weight * g["fill_norm"] -
            g["edge_penalty_val"]
        )
        g = g.sort_values(by=["z_score", area_col, circularity_col], ascending=[False, False, False])
        selected_rows.append(g.iloc[0])

    return pd.DataFrame(selected_rows).reset_index(drop=True)


def apply_time_aware_final_area_filter(
    df: pd.DataFrame,
    pixel_size_um: float = 0.162,
    area_px_col: str = "area_px",
    area_um2_col: str = "area_um2",
    time_col: str = "true_time_min",
    early_time_max: float = 20,
    mid_time_max: float = 40,
    early_min_area_um2: float = 120,
    mid_min_area_um2: float = 160,
    late_min_area_um2: float = 180,
):
    if df is None or df.empty:
        print("No data to filter.")
        return df.copy()

    out = df.copy()
    if area_um2_col not in out.columns:
        if area_px_col not in out.columns:
            raise ValueError(f"Missing both '{area_um2_col}' and '{area_px_col}'")
        out[area_um2_col] = out[area_px_col] * (pixel_size_um ** 2)
    if time_col not in out.columns:
        raise ValueError(f"Missing required column '{time_col}'")

    keep_early = (out[time_col] < early_time_max) & (out[area_um2_col] >= early_min_area_um2)
    keep_mid = (out[time_col] >= early_time_max) & (out[time_col] < mid_time_max) & (out[area_um2_col] >= mid_min_area_um2)
    keep_late = (out[time_col] >= mid_time_max) & (out[area_um2_col] >= late_min_area_um2)

    filtered = out[keep_early | keep_mid | keep_late].copy()
    print("Rows before final time-aware filter:", len(out))
    print("Rows after final time-aware filter:", len(filtered))
    return filtered


def score_2d_objects_for_qc(
    objects_df: pd.DataFrame,
    label_img: np.ndarray,
    nuc_prob_img: np.ndarray,
    pixel_size_um: float = 0.162,
    area_px_col: str = "area_px",
    label_col: str = "label",
    bbox_min_row_col: str = "bbox_min_row",
    bbox_min_col_col: str = "bbox_min_col",
    bbox_max_row_col: str = "bbox_max_row",
    bbox_max_col_col: str = "bbox_max_col",
    circularity_col: str = "circularity",
    edge_col: str = "is_edge_touching",
    min_area_um2: float = 120,
    max_area_um2: float = 600,
    min_fill_fraction: float = 0.60,
    min_circularity: float = 0.55,
    min_mean_prob: float = 0.40,
    min_max_prob: float = 0.60,
    qc_score_threshold: float = 0.35,
):
    if objects_df is None or objects_df.empty:
        print("No objects to score.")
        return objects_df.copy()

    out = objects_df.copy()

    if "area_um2" not in out.columns:
        if area_px_col not in out.columns:
            raise ValueError(f"Missing required column '{area_px_col}'")
        out["area_um2"] = out[area_px_col] * (pixel_size_um ** 2)

    needed_bbox = [bbox_min_row_col, bbox_min_col_col, bbox_max_row_col, bbox_max_col_col]
    if all(c in out.columns for c in needed_bbox):
        bbox_h = (out[bbox_max_row_col] - out[bbox_min_row_col]).clip(lower=1)
        bbox_w = (out[bbox_max_col_col] - out[bbox_min_col_col]).clip(lower=1)
        out["bbox_area_px"] = bbox_h * bbox_w
        out["fill_fraction"] = out[area_px_col] / out["bbox_area_px"]
    else:
        out["fill_fraction"] = np.nan

    if edge_col not in out.columns:
        image_h, image_w = label_img.shape
        if all(c in out.columns for c in needed_bbox):
            out[edge_col] = (
                (out[bbox_min_row_col] <= 0) |
                (out[bbox_min_col_col] <= 0) |
                (out[bbox_max_row_col] >= image_h - 1) |
                (out[bbox_max_col_col] >= image_w - 1)
            )
        else:
            out[edge_col] = False

    if circularity_col not in out.columns:
        out[circularity_col] = np.nan

    mean_probs = []
    max_probs = []
    for _, row in out.iterrows():
        lab = row[label_col]
        mask = label_img == lab
        if mask.sum() == 0:
            mean_probs.append(np.nan)
            max_probs.append(np.nan)
            continue
        vals = nuc_prob_img[mask]
        mean_probs.append(float(np.mean(vals)))
        max_probs.append(float(np.max(vals)))

    out["mean_nuc_prob"] = mean_probs
    out["max_nuc_prob"] = max_probs

    def norm_clip(series, low, high):
        s = series.astype(float)
        outv = (s - low) / (high - low)
        return outv.clip(0, 1)

    area_center = (min_area_um2 + max_area_um2) / 2
    area_halfwidth = (max_area_um2 - min_area_um2) / 2
    out["area_score"] = 1 - ((out["area_um2"] - area_center).abs() / area_halfwidth)
    out["area_score"] = out["area_score"].clip(0, 1)

    out["fill_score"] = norm_clip(out["fill_fraction"].fillna(0), min_fill_fraction, 1.0)
    out["circ_score"] = norm_clip(out[circularity_col].fillna(0), min_circularity, 1.0)
    out["mean_prob_score"] = norm_clip(out["mean_nuc_prob"].fillna(0), min_mean_prob, 1.0)
    out["max_prob_score"] = norm_clip(out["max_nuc_prob"].fillna(0), min_max_prob, 1.0)
    out["edge_penalty"] = out[edge_col].astype(float)

    out["qc_score"] = (
        0.20 * out["area_score"] +
        0.20 * out["fill_score"] +
        0.15 * out["circ_score"] +
        0.25 * out["mean_prob_score"] +
        0.20 * out["max_prob_score"] -
        0.50 * out["edge_penalty"]
    )

    out["qc_keep"] = (
        (out["area_um2"] >= min_area_um2) &
        (out["area_um2"] <= max_area_um2) &
        (out["fill_fraction"] >= min_fill_fraction) &
        (out["mean_nuc_prob"] >= min_mean_prob) &
        (out["max_nuc_prob"] >= min_max_prob) &
        (~out[edge_col]) &
        (out["qc_score"] >= qc_score_threshold)
    )
    return out


def choose_time_column(df: pd.DataFrame):
    if df is None:
        return None, None
    if "true_time_min" in df.columns:
        return "true_time_min", "Time (min)"
    if "t" in df.columns:
        return "t", "Time index"
    return None, None


def plot_counts_by_time(df: pd.DataFrame, title: str = "Object count by time"):
    if df is None or df.empty:
        print("No data to plot.")
        return
    xcol, xlabel = choose_time_column(df)
    if xcol is None:
        print("No time column found.")
        return
    plot_df = df.groupby(xcol).size().reset_index(name="count")
    plt.figure(figsize=(8, 4))
    plt.plot(plot_df[xcol], plot_df["count"], marker="o")
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_area_distribution(df: pd.DataFrame, area_col: str = "area_um2", title: str = "Area distribution"):
    if df is None or df.empty:
        print("No data to plot.")
        return
    vals = pd.to_numeric(df[area_col], errors="coerce").dropna()
    plt.figure(figsize=(8, 4))
    plt.hist(vals, bins=50)
    plt.xlabel(area_col)
    plt.ylabel("Frequency")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_selected_best_z_results(
    df: pd.DataFrame,
    pixel_size_um: float,
    n_fov: int = 6,
    serpentine: bool = True,
    area_px_col: str = "area_px",
    area_um2_col: str = "area_um2",
    title: str = "Best-Z max cross-sectional area vs time",
):
    if df is None or df.empty:
        print("No data to plot.")
        return df

    out = df.copy()
    if area_um2_col not in out.columns:
        if area_px_col not in out.columns:
            raise ValueError(f"Missing both '{area_um2_col}' and '{area_px_col}'")
        out[area_um2_col] = out[area_px_col] * (pixel_size_um ** 2)

    if "true_time_min" not in out.columns:
        out = reconstruct_true_time_from_x(out, n_fov=n_fov, serpentine=serpentine)

    plt.figure(figsize=(9, 4.5))
    plt.scatter(out["true_time_min"], out[area_um2_col], s=20, alpha=0.6)
    plt.xlabel("Time (min)")
    plt.ylabel("Max cross-sectional area (µm²)")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return out


def plot_grouped_object_cleaning_diagnostics(
    raw_df: pd.DataFrame,
    cleaned_df: pd.DataFrame,
    t_value: int,
    pixel_size_um: float,
    area_px_col: str = "area_px",
    z_col: str = "z",
):
    if raw_df is None or raw_df.empty:
        print("raw_df is empty.")
        return
    if cleaned_df is None or cleaned_df.empty:
        print("cleaned_df is empty.")
        return

    raw_t = raw_df[raw_df["t"] == t_value].copy()
    clean_t = cleaned_df[cleaned_df["t"] == t_value].copy()
    if raw_t.empty or clean_t.empty:
        print(f"No rows for t={t_value}")
        return

    raw_t["area_um2"] = raw_t[area_px_col] * (pixel_size_um ** 2)
    clean_t["area_um2"] = clean_t[area_px_col] * (pixel_size_um ** 2)

    raw_counts = raw_t.groupby(z_col).size().reset_index(name="n_objects")
    clean_counts = clean_t.groupby(z_col).size().reset_index(name="n_objects")

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes[0, 0].plot(raw_counts[z_col], raw_counts["n_objects"], marker="o")
    axes[0, 0].set_title(f"Objects per Z before cleaning at t={t_value}")
    axes[0, 0].set_xlabel("Z")
    axes[0, 0].set_ylabel("Number of grouped objects")

    axes[0, 1].plot(clean_counts[z_col], clean_counts["n_objects"], marker="o")
    axes[0, 1].set_title(f"Objects per Z after cleaning at t={t_value}")
    axes[0, 1].set_xlabel("Z")
    axes[0, 1].set_ylabel("Number of grouped objects")

    for z_val, g in raw_t.groupby(z_col):
        axes[1, 0].scatter([z_val] * len(g), g["area_um2"], s=10, alpha=0.5)
    axes[1, 0].set_title(f"Per-slice object areas before cleaning at t={t_value}")
    axes[1, 0].set_xlabel("Z")
    axes[1, 0].set_ylabel("Area (µm²)")

    for z_val, g in clean_t.groupby(z_col):
        axes[1, 1].scatter([z_val] * len(g), g["area_um2"], s=10, alpha=0.5)
    axes[1, 1].set_title(f"Per-slice object areas after cleaning at t={t_value}")
    axes[1, 1].set_xlabel("Z")
    axes[1, 1].set_ylabel("Area (µm²)")

    plt.tight_layout()
    plt.show()


def plot_valid_nuclei_diagnostics(df: pd.DataFrame, t_value: int, pixel_size_um: float, area_px_col: str = "area_px", z_col: str = "z"):
    if df is None or df.empty:
        print("No valid nuclei to plot.")
        return

    plot_df = df[df["t"] == t_value].copy()
    if plot_df.empty:
        print(f"No valid nuclei found for t={t_value}")
        return

    if "area_um2" not in plot_df.columns:
        plot_df["area_um2"] = plot_df[area_px_col] * (pixel_size_um ** 2)

    counts = plot_df.groupby(z_col).size().reset_index(name="n_objects")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(counts[z_col], counts["n_objects"], marker="o")
    axes[0].set_title(f"Valid nuclei per Z at t={t_value}")
    axes[0].set_xlabel("Z")
    axes[0].set_ylabel("Number of valid nuclei")

    for z_val, g in plot_df.groupby(z_col):
        axes[1].scatter([z_val] * len(g), g["area_um2"], s=12, alpha=0.6)
    axes[1].set_title(f"Valid nucleus areas per Z at t={t_value}")
    axes[1].set_xlabel("Z")
    axes[1].set_ylabel("Area (µm²)")
    plt.tight_layout()
    plt.show()


def plot_nc_ratio_from_halo(df, title="N/C ratio vs time (final filtered nuclei)"):
    if df is None or df.empty:
        print("No data to plot.")
        return

    required = ["true_time_min", "nc_ratio_fraction"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    plot_df = df.dropna(subset=["true_time_min", "nc_ratio_fraction"]).copy()
    plot_df = plot_df.sort_values("true_time_min")

    plt.figure(figsize=(8, 4.5))
    plt.scatter(plot_df["true_time_min"], plot_df["nc_ratio_fraction"], s=15, alpha=0.5)

    smooth = plot_df["nc_ratio_fraction"].rolling(
        window=max(10, len(plot_df) // 50),
        min_periods=1
    ).mean()

    plt.plot(plot_df["true_time_min"], smooth, linewidth=2)
    plt.xlabel("Time (min)")
    plt.ylabel("N / (N + C)")
    plt.title(title)
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.show()


def open_nuclear_only_napari(seg_class, nucleus_instance, t_idx: int = 0, show_all_z: bool = True):
    import napari

    nuclear_labels = None
    roi_mask = None

    if seg_class is not None:
        arr = seg_class
        if arr.ndim == 5:  # (T, Z, C, Y, X)
            arr = arr[t_idx]
        elif arr.ndim == 4 and nucleus_instance is not None and getattr(nucleus_instance, "ndim", None) == 4:
            arr = arr[t_idx]

        if arr.ndim == 4 and arr.shape[1] <= 10:
            nuclear_labels = np.argmax(arr, axis=1).astype(np.uint8)
        elif arr.ndim == 3:
            nuclear_labels = arr.astype(np.uint8)
        else:
            raise ValueError(f"Unsupported seg_class shape after slicing: {arr.shape}")

    if nucleus_instance is not None:
        arr = nucleus_instance
        if arr.ndim == 4:
            arr = arr[t_idx]
        elif arr.ndim != 3:
            raise ValueError(f"Unsupported nucleus_instance shape: {arr.shape}")
        roi_mask = arr

    if not show_all_z:
        if nuclear_labels is not None and nuclear_labels.ndim == 3:
            nuclear_labels = nuclear_labels[nuclear_labels.shape[0] // 2]
        if roi_mask is not None and roi_mask.ndim == 3:
            roi_mask = roi_mask[roi_mask.shape[0] // 2]

    viewer = napari.Viewer()
    if nuclear_labels is not None:
        viewer.add_labels(nuclear_labels, name="nuclear_class_labels")
    if roi_mask is not None:
        viewer.add_labels(roi_mask, name="nuclear_roi_mask")
    return viewer


## 3. Load bundle tables and masks

In [ ]:

bundle_dir = BUNDLE_DIR
assert bundle_dir.exists(), f"Bundle directory not found: {bundle_dir}"

manifest_path = bundle_dir / "manifest.json"
config_path = bundle_dir / "config" / "pipeline_config.json"
tables_dir = bundle_dir / "tables"
masks_dir = bundle_dir / "masks"

manifest = load_json(manifest_path) if manifest_path.exists() else {}
config = load_json(config_path) if config_path.exists() else {}

print("Bundle:", bundle_dir.resolve())
print("Manifest found:", manifest_path.exists())
print("Config found:", config_path.exists())

table_names = [
    "segmentation_index",
    "segmentation_index_portable",
    "plane_objects",
    "grouped_z_objects",
    "best_z_nuclei",
    "best_z_nuclei_cleaned",
    "best_z_nuclei_with_exclusion",
    "best_z_nuclei_timed",
    "tracked_nuclei",
    "halo_analysis",
]

tables = {}
table_sources = {}
for name in table_names:
    df, src = load_table_prefer_parquet_csv_pickle(tables_dir / name)
    tables[name] = df
    table_sources[name] = src

mask_files = {
    "segmentation_class": masks_dir / "segmentation_class_hyperstack.tif",
    "segmentation_label": masks_dir / "segmentation_label_hyperstack.tif",
    "nucleus_instance": masks_dir / "nucleus_instance_hyperstack.tif",
    "droplet_instance": masks_dir / "droplet_instance_hyperstack.tif",
    "segmentation_prob": masks_dir / "segmentation_prob_hyperstack.tif",
}

mask_arrays = {name: safe_read_tiff(path) if path.exists() else None for name, path in mask_files.items()}
raw_img = safe_read_tiff(Path(RAW_IMAGE_PATH)) if RAW_IMAGE_PATH is not None else None


## 4. Choose active tables

In [ ]:

seg_index_df = tables.get("segmentation_index_portable") or tables.get("segmentation_index")
plane_objects_df = tables.get("plane_objects")
grouped_z_df = tables.get("grouped_z_objects")
tracked_df = tables.get("tracked_nuclei")
halo_df = tables.get("halo_analysis")

best_z_source_name = None
for candidate in ["best_z_nuclei_cleaned", "best_z_nuclei_timed", "best_z_nuclei_with_exclusion", "best_z_nuclei"]:
    if tables.get(candidate) is not None:
        best_z_df = tables[candidate].copy()
        best_z_source_name = candidate
        break
else:
    best_z_df = None

if best_z_df is not None:
    best_z_df = ensure_area_um2(best_z_df, PIXEL_SIZE_UM)
    if {"t", "centroid_x_px"}.issubset(best_z_df.columns):
        best_z_df = reconstruct_true_time_from_x(best_z_df, n_fov=N_FOV, serpentine=SERPENTINE)

print("Active best_z source:", best_z_source_name)


## 5. Quick summary

In [ ]:

for name in table_names:
    df = tables.get(name)
    src = table_sources.get(name)
    print(f"{name}: {'missing' if df is None else df.shape} | source={src}")

print("\nMask availability:")
for name, arr in mask_arrays.items():
    if arr is None:
        print(name, "missing")
    else:
        print(name, getattr(arr, "shape", None), getattr(arr, "dtype", None))


## 6. Inspect the main working table

In [ ]:
describe_df(best_z_df, f"best_z_df ({best_z_source_name})")

## 7. Core QC plots

In [ ]:

if best_z_df is not None and not best_z_df.empty:
    plot_counts_by_time(best_z_df, f"Counts by time: {best_z_source_name}")
    plot_area_distribution(best_z_df, area_col="area_um2", title=f"Area distribution: {best_z_source_name}")
    plot_selected_best_z_results(
        best_z_df,
        pixel_size_um=PIXEL_SIZE_UM,
        n_fov=N_FOV,
        serpentine=SERPENTINE,
        title=f"Max cross-sectional area vs time: {best_z_source_name}",
    )


## 8. Grouped-object cleaning, 3D grouping, valid nuclei, and final best-Z selection

In [ ]:

grouped_z_clean_df = None
grouped_valid_input_df = None
valid_nuclei_df = None
best_z_valid_df = None
best_z_valid_timeaware_df = None

if grouped_z_df is None or grouped_z_df.empty:
    print("grouped_z_df not available.")
else:
    grouped_z_clean_df = clean_grouped_objects_before_best_z(
        grouped_z_df,
        min_area_px=MIN_AREA_PX_GROUP,
        min_fill_fraction=MIN_FILL_FRACTION_GROUP,
        min_circularity=None,
        max_area_um2=600,
        pixel_size_um=PIXEL_SIZE_UM,
        remove_edge_touching=True,
        edge_buffer_px=EDGE_BUFFER_PX,
    )

    grouped_valid_input_df = group_objects_across_z(
        grouped_z_clean_df,
        max_dist_px=GROUP_MAX_DIST_PX,
    )

    valid_nuclei_df = keep_valid_nuclei(
        grouped_valid_input_df,
        min_z_slices=MIN_Z_SLICES_VALID,
    )

    best_z_valid_df = select_best_z_by_composite_score(valid_nuclei_df)
    best_z_valid_df = ensure_area_um2(best_z_valid_df, PIXEL_SIZE_UM)
    best_z_valid_df = reconstruct_true_time_from_x(best_z_valid_df, n_fov=N_FOV, serpentine=SERPENTINE)

    best_z_valid_timeaware_df = apply_time_aware_final_area_filter(
        best_z_valid_df,
        pixel_size_um=PIXEL_SIZE_UM,
        time_col="true_time_min",
        early_time_max=20,
        mid_time_max=40,
        early_min_area_um2=120,
        mid_min_area_um2=160,
        late_min_area_um2=180,
    )

    describe_df(best_z_valid_timeaware_df, "best_z_valid_timeaware_df")


## 9. Diagnostic plots for grouped objects and valid nuclei

In [ ]:

if grouped_z_df is not None and grouped_z_clean_df is not None and not grouped_z_df.empty and not grouped_z_clean_df.empty:
    for t_val in [0, 3, 5, 9]:
        plot_grouped_object_cleaning_diagnostics(
            raw_df=grouped_z_df,
            cleaned_df=grouped_z_clean_df,
            t_value=t_val,
            pixel_size_um=PIXEL_SIZE_UM,
        )

if valid_nuclei_df is not None and not valid_nuclei_df.empty:
    for t_val in [0, 3, 5, 9]:
        plot_valid_nuclei_diagnostics(
            valid_nuclei_df,
            t_value=t_val,
            pixel_size_um=PIXEL_SIZE_UM,
        )


## 10. Final best-Z area plot

In [ ]:

if best_z_valid_timeaware_df is not None and not best_z_valid_timeaware_df.empty:
    best_z_valid_timeaware_df = plot_selected_best_z_results(
        best_z_valid_timeaware_df,
        pixel_size_um=PIXEL_SIZE_UM,
        n_fov=N_FOV,
        serpentine=SERPENTINE,
        title="Valid nuclei only: best-Z max cross-sectional area vs time (time-aware filtered)",
    )


## 11. Final N/C ratio plot from halo table

In [ ]:

halo_bestz_df = None

if halo_df is None or halo_df.empty:
    print("halo_df not available.")
elif best_z_valid_timeaware_df is None or best_z_valid_timeaware_df.empty:
    print("best_z_valid_timeaware_df not available.")
else:
    best_z_keys = best_z_valid_timeaware_df[["t", "z", "label"]].drop_duplicates().copy()
    halo_bestz_df = halo_df.merge(best_z_keys, on=["t", "z", "label"], how="inner").copy()

    print("Halo rows before:", len(halo_df))
    print("Rows after best-Z filtering:", len(halo_bestz_df))

    if not halo_bestz_df.empty:
        plot_nc_ratio_from_halo(halo_bestz_df)


## 12. 2D object QC scoring for one selected plane

In [ ]:

QC_T_IDX = 0
QC_Z_IDX = 0

qc_df = None

seg_label_arr = mask_arrays.get("segmentation_label")
seg_prob_arr = mask_arrays.get("segmentation_prob")

if plane_objects_df is None or plane_objects_df.empty:
    print("plane_objects_df not available.")
elif seg_label_arr is None or seg_prob_arr is None:
    print("Need segmentation_label and segmentation_prob masks for QC scoring.")
else:
    plane_df = plane_objects_df[(plane_objects_df["t"] == QC_T_IDX) & (plane_objects_df["z"] == QC_Z_IDX)].copy()

    label_img = None
    prob_img = None

    # segmentation_label is expected to be (T, Z, Y, X) or memmap-compatible equivalent
    if getattr(seg_label_arr, "ndim", None) == 4:
        label_img = np.asarray(seg_label_arr[QC_T_IDX, QC_Z_IDX])
    elif getattr(seg_label_arr, "ndim", None) == 3:
        label_img = np.asarray(seg_label_arr[QC_Z_IDX])

    # segmentation_prob is expected to be (T, Z, Y, X) for nuclear probability
    if getattr(seg_prob_arr, "ndim", None) == 4:
        prob_img = np.asarray(seg_prob_arr[QC_T_IDX, QC_Z_IDX])
    elif getattr(seg_prob_arr, "ndim", None) == 3:
        prob_img = np.asarray(seg_prob_arr[QC_Z_IDX])

    if plane_df.empty:
        print(f"No plane_objects rows for t={QC_T_IDX}, z={QC_Z_IDX}")
    elif label_img is None or prob_img is None:
        print("Could not extract label/probability slice for QC scoring.")
    else:
        qc_df = score_2d_objects_for_qc(
            objects_df=plane_df,
            label_img=label_img,
            nuc_prob_img=prob_img,
            pixel_size_um=PIXEL_SIZE_UM,
            min_area_um2=QC_MIN_AREA_UM2,
            max_area_um2=QC_MAX_AREA_UM2,
            min_fill_fraction=QC_MIN_FILL_FRACTION,
            min_circularity=QC_MIN_CIRCULARITY,
            min_mean_prob=QC_MIN_MEAN_PROB,
            min_max_prob=QC_MIN_MAX_PROB,
            qc_score_threshold=QC_SCORE_THRESHOLD,
        )
        describe_df(qc_df, f"qc_df t={QC_T_IDX}, z={QC_Z_IDX}")


## 13. QC score plots

In [ ]:

if qc_df is not None and not qc_df.empty:
    plt.figure(figsize=(7, 4))
    plt.hist(qc_df["qc_score"].dropna(), bins=30)
    plt.xlabel("QC score")
    plt.ylabel("Count")
    plt.title("2D object QC score distribution")
    plt.tight_layout()
    plt.show()

    accepted = qc_df[qc_df["qc_keep"]]
    rejected = qc_df[~qc_df["qc_keep"]]

    print("Accepted:", len(accepted))
    print("Rejected:", len(rejected))

    display(qc_df[[
        "label", "area_um2", "fill_fraction", "circularity",
        "mean_nuc_prob", "max_nuc_prob", "qc_score", "qc_keep"
    ]].sort_values("qc_score", ascending=False).head(20))


## 14. Slim napari viewer

In [ ]:

if AUTO_OPEN_NAPARI:
    viewer = open_nuclear_only_napari(
        seg_class=mask_arrays["segmentation_class"],
        nucleus_instance=mask_arrays["nucleus_instance"],
        t_idx=NAPARI_T_IDX,
        show_all_z=NAPARI_SHOW_ALL_Z,
    )


## 15. Useful quick checks

In [ ]:

tables_to_check = {
    "best_z_valid_timeaware_df": best_z_valid_timeaware_df if "best_z_valid_timeaware_df" in globals() else None,
    "best_z_valid_df": best_z_valid_df if "best_z_valid_df" in globals() else None,
    "halo_df": halo_df if "halo_df" in globals() else None,
    "tracked_df": tracked_df if "tracked_df" in globals() else None,
}

for name, table in tables_to_check.items():
    print(f"\n{name}")
    if table is None:
        print("missing")
    else:
        print(table.columns.tolist())
